# Rozdział 10 — Walidacja, odporność i reprodukowalność

Eksperymenty E10.1–E10.8 odtwarzają figury i tabele rozdziału. Dane wejściowe
są lokalne. Sekcje E10.1–E10.4 dotyczą metodologii i danych rozwojowych.
E10.5 otwiera odłożony rekord 200 przy konfiguracji z rozdziału 9.
Następne sekcje opisują wynik tego samego przebiegu.

Rewizja D01 (12 listopada 2026): położenia adnotacji EKG pochodzą z oryginalnych plików WFDB `.atr`, v1.0.0. Próbki sygnału pozostają zgodne z wycinkami CPSB. Ślad korekty: `chapter_notes/korekta_adnotacji_D01.md`.
Protokół `ch10-v0.2-D01` dotyczy regresji na wcześniej znanym rekordzie 200; ponowne wykonanie nie jest nową niezależną walidacją.


In [1]:
from copy import deepcopy
import json
from hashlib import sha256
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

root = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "config/ch10_protocol.json").is_file())
sys.path.insert(0, str(root / "src"))
from bspsbook.ch09 import LABELS, checksum, detect, load_development, write_csv
from bspsbook.ch10 import (
    aggregate, choose_failure, compare, count_metrics, didactic_checks, environment,
    evaluate, inside_rr, load_handoff, load_locked,
)
from bspsbook.qrs_core import fit_baseline_state, make_semisynthetic_variant, match_events_1to1

protocol_path = root / "config/ch10_protocol.json"
protocol = json.loads(protocol_path.read_text(encoding="utf-8"))
frozen, state = load_handoff(root, protocol)
methods = protocol["methods"]
colors = {"baseline": "#095b89", "matched_filter": "#a44b24"}
figures, outputs = root / "figures/generated", root / "results"
figures.mkdir(parents=True, exist_ok=True)
outputs.mkdir(exist_ok=True)
plt.rcParams.update({
    "font.size": 10, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": .20, "legend.frameon": False,
    "svg.hashsalt": "CPSB-ch10",
})

def save(fig, slug):
    fig.savefig(figures / (slug + ".png"), dpi=175, metadata={"Software": "CPSB"})
    fig.savefig(figures / (slug + ".svg"), metadata={"Date": None})
    plt.close(fig)

def show_rows(rows, keys):
    print(" | ".join(keys))
    for row in rows:
        print(" | ".join(
            f"{row[key]:.4f}" if isinstance(row[key], float)
            else "—" if row[key] is None else str(row[key]) for key in keys
        ))

## E10.1 — Kontrole etapowe i dwie role F1

In [2]:
example = match_events_1to1(np.array([0, 10]), np.array([6, 14]), 6)
assert example.errors_samples.tolist() == [6, 4]
assert match_events_1to1(np.array([100]), np.array([118]), 18).errors_samples.tolist() == [18]
assert match_events_1to1(np.array([100]), np.array([119]), 18).errors_samples.size == 0
assert count_metrics(0, 0, 3)["f1"] == 0 and count_metrics(0, 0, 0)["f1"] is None
mask = np.array([True, True, False, True, True])
assert inside_rr(np.array([0, 1, 3, 4]), mask, 1) == [1, 1]
examples = [count_metrics(90, 10, 0), count_metrics(90, 0, 10)]
assert examples[0]["f1"] == examples[1]["f1"]
show_rows(examples, ["tp", "fp", "fn", "sensitivity", "ppv", "f1"])

train, validation, audit = load_development(root)
assert {f["metadata"]["record_id"] for f in train}.isdisjoint(
    {f["metadata"]["record_id"] for f in validation}
)
print("Zweryfikowano dane rozwojowe, pliki przekazania, tolerancję, ciągłość RR i mianowniki.")

tp | fp | fn | sensitivity | ppv | f1
90 | 10 | 0 | 1.0000 | 0.9000 | 0.9474
90 | 0 | 10 | 0.9000 | 1.0000 | 0.9474
Zweryfikowano dane rozwojowe, pliki przekazania, tolerancję, ciągłość RR i mianowniki.


## E10.2 — Zmiana długości okna energii przy ustalonej procedurze

In [3]:
sensitivity = []
for width in protocol["sensitivity"]["energy_window_ms"]:
    cfg = deepcopy(frozen)
    cfg["baseline"]["energy_window_ms"] = width
    new_state = {**state, "baseline": fit_baseline_state(train, cfg)}
    if width == frozen["baseline"]["energy_window_ms"]:
        for key in ("energy_center_train", "energy_scale_train"):
            np.testing.assert_allclose(new_state["baseline"][key], state["baseline"][key],
                                       rtol=1e-12, atol=0)
        new_state = state

    items = [evaluate(f, "baseline", cfg, new_state)[1] for f in validation]
    row = aggregate(items)
    sensitivity.append({
        "energy_window_ms": width,
        "energy_window_samples": new_state["baseline"]["energy_window_samples_by_fs"]["360"],
        "energy_center_mV2": new_state["baseline"]["energy_center_train"],
        "energy_scale_mV2": new_state["baseline"]["energy_scale_train"],
        **row,
    })
write_csv(outputs / "ch10_sensitivity.csv", sensitivity)
show_rows(sensitivity, ["energy_window_ms", "energy_window_samples", "tp", "fp", "fn",
                        "f1", "timing_mae_ms", "result_coverage"])
fig, axes = plt.subplots(2, 1, sharex=True, figsize=(7.7, 5.1), layout="constrained")
widths = [r["energy_window_ms"] for r in sensitivity]
for key, label, marker in (("sensitivity", "Se", "o"), ("ppv", "PPV", "s")):
    axes[0].plot(widths, [r[key] for r in sensitivity], marker=marker, ms=4, label=label)
axes[0].set(ylabel="Miara zdarzeniowa [–]")
axes[0].legend(loc="lower right", ncol=2)
axes[1].plot(widths, [r["timing_mae_ms"] for r in sensitivity], "o-", ms=4, color=colors["baseline"])
axes[1].set(ylabel="MAE czasu [ms]", xlabel="Nominalna długość okna energii [ms]", xticks=widths)
for ax in axes:
    ax.axvline(frozen["baseline"]["energy_window_ms"], ls="--", lw=.9, color="#777777")
save(fig, "ch10_01_czulosc")

energy_window_ms | energy_window_samples | tp | fp | fn | f1 | timing_mae_ms | result_coverage
60 | 23 | 248 | 6 | 0 | 0.9880 | 0.2352 | 1.0000
90 | 33 | 248 | 4 | 0 | 0.9920 | 0.2352 | 1.0000
120 | 43 | 248 | 3 | 0 | 0.9940 | 0.2352 | 1.0000
160 | 59 | 248 | 4 | 0 | 0.9920 | 0.2352 | 1.0000
200 | 73 | 248 | 3 | 0 | 0.9940 | 0.2352 | 1.0000


## E10.3 — Odporność na kontrolowane zmiany wejścia

In [4]:
stress_plan = protocol["stress"]
normal = next(f for f in validation if f["fragment_id"] == stress_plan["fragment_id"])
seeds = stress_plan["seeds"]
levels = stress_plan["noise_sd_mV"]
stress_rows = []

def stress_run(variant, level, seed):
    fragment = make_semisynthetic_variant(normal, variant, level, seed, frozen) if level else normal
    np.testing.assert_array_equal(fragment["r_samples"], normal["r_samples"])
    entries = [evaluate(fragment, method, frozen, state) for method in methods]
    np.testing.assert_array_equal(entries[0][0].evaluation_mask, entries[1][0].evaluation_mask)
    return [{"variant": variant, "level": level, "seed": seed, **item["summary"]}
            for _, item in entries]

for level in levels:
    for seed in seeds if level else [seeds[0]]:
        stress_rows.extend(stress_run("white_noise_mV", level, seed))
for variant, level in stress_plan["scenarios"]:
    stress_rows.extend(stress_run(variant, level, seeds[0]))
write_csv(outputs / "ch10_stress.csv", stress_rows)
show_rows(
    [r for r in stress_rows if r["variant"] != "white_noise_mV"],
    ["variant", "level", "method", "tp", "fp", "fn", "reference_evaluated", "f1", "result_coverage"],
)

fig, axes = plt.subplots(3, 1, sharex=True, figsize=(7.7, 6.5), layout="constrained")
for ax, key, label in zip(
    axes, ("f1", "timing_mae_ms", "result_coverage"),
    ("F1 [–]", "MAE czasu [ms]", "Pokrycie HR [–]"),
):
    for method in methods:
        groups = [[r[key] for r in stress_rows
                   if r["method"] == method and r["variant"] == "white_noise_mV"
                   and r["level"] == level and r[key] is not None] for level in levels]
        med = [np.median(g) if g else np.nan for g in groups]
        lo = [min(g) if g else np.nan for g in groups]
        hi = [max(g) if g else np.nan for g in groups]
        ax.plot(levels, med, marker="o", ms=4, color=colors[method], label=LABELS[method],
                ls="-" if method == "baseline" else "--")
        ax.fill_between(levels, lo, hi, alpha=.16, color=colors[method])
    ax.set_ylabel(label)
axes[0].legend(loc="lower left", ncol=2)
axes[-1].set(xlabel="Odchylenie standardowe dodanego szumu [mV]", xticks=levels)
save(fig, "ch10_02_szum")

variant | level | method | tp | fp | fn | reference_evaluated | f1 | result_coverage
baseline_drift_mV | 0.5000 | baseline | 122 | 0 | 0 | 122 | 1.0000 | 1.0000
baseline_drift_mV | 0.5000 | matched_filter | 122 | 0 | 0 | 122 | 1.0000 | 1.0000
impulse_artifacts_mV | 1.0000 | baseline | 122 | 6 | 0 | 122 | 0.9760 | 1.0000
impulse_artifacts_mV | 1.0000 | matched_filter | 122 | 6 | 0 | 122 | 0.9760 | 1.0000
gap_s | 3.0000 | baseline | 117 | 0 | 0 | 117 | 1.0000 | 0.8889
gap_s | 3.0000 | matched_filter | 117 | 0 | 0 | 117 | 1.0000 | 0.8889
saturation_s | 2.0000 | baseline | 118 | 0 | 0 | 118 | 1.0000 | 0.8889
saturation_s | 2.0000 | matched_filter | 118 | 0 | 0 | 118 | 1.0000 | 0.8889
polarity | -1.0000 | baseline | 122 | 0 | 0 | 122 | 1.0000 | 1.0000
polarity | -1.0000 | matched_filter | 120 | 0 | 2 | 122 | 0.9917 | 1.0000
qrs_broadening | 1.4000 | baseline | 122 | 0 | 0 | 122 | 1.0000 | 1.0000
qrs_broadening | 1.4000 | matched_filter | 122 | 0 | 0 | 122 | 1.0000 | 1.0000


## E10.4 — Błąd częstości i pokrycie przy różnych progach SQI

In [5]:
diag = protocol["quality_diagnostic"]
diagnostic_rows = []
for original in validation:
    assert original["metadata"]["record_id"] == diag["record"]
    fragment = make_semisynthetic_variant(original, "white_noise_mV", diag["noise_sd_mV"], seeds[0], frozen)
    for method in methods:
        result = detect(fragment, method, frozen, state)
        for threshold in diag["sqi_min"]:
            cfg = deepcopy(frozen)
            cfg["quality"]["sqi_min"] = threshold
            diagnostic_rows.append({
                "origin": original["fragment_id"], "sqi_min": threshold,
                **compare(fragment, result, cfg)["summary"],
            })
for method in methods:
    for original in validation:
        part = [r for r in diagnostic_rows if r["origin"] == original["fragment_id"] and r["method"] == method]
        assert len({(r["tp"], r["fp"], r["fn"], r["windows"]) for r in part}) == 1
        counts = [r["available_windows"] for r in part]
        assert counts == sorted(counts, reverse=True)
write_csv(outputs / "ch10_quality_curve.csv", diagnostic_rows)
show_rows(
    [r for r in diagnostic_rows if r["sqi_min"] in (.55, .70, .85)],
    ["origin", "method", "sqi_min", "windows", "available_windows", "window_hr_mae_bpm"],
)
fig, axes = plt.subplots(2, 2, sharex=True, sharey="row", figsize=(8.4, 5.4), layout="constrained")
for col, original in enumerate(validation):
    name = original["fragment_id"]
    for method in methods:
        part = [r for r in diagnostic_rows if r["origin"] == name and r["method"] == method]
        for row, key in enumerate(("result_coverage", "window_hr_mae_bpm")):
            axes[row, col].plot(
                [r["sqi_min"] for r in part], [r[key] for r in part],
                "o-" if method == "baseline" else "s--", ms=4,
                color=colors[method], label=LABELS[method],
            )
    axes[0, col].set(title=name, ylim=(-.04, 1.06))
    axes[1, col].set(xlabel="Próg SQI [–]")
    for ax in axes[:, col]:
        ax.axvline(frozen["quality"]["sqi_min"], ls=":", lw=.9, color="#777777")
axes[0, 0].set_ylabel("Pokrycie HR [–]")
axes[1, 0].set_ylabel("MAE częstości [1/min]")
axes[0, 1].legend(loc="upper right", fontsize=9)
save(fig, "ch10_03_pokrycie")

origin | method | sqi_min | windows | available_windows | window_hr_mae_bpm
105_normal | baseline | 0.5500 | 9 | 9 | 30.3917
105_normal | baseline | 0.7000 | 9 | 8 | 23.5266
105_normal | baseline | 0.8500 | 9 | 0 | —
105_normal | matched_filter | 0.5500 | 9 | 9 | 0.1430
105_normal | matched_filter | 0.7000 | 9 | 9 | 0.1430
105_normal | matched_filter | 0.8500 | 9 | 0 | —
105_artifact | baseline | 0.5500 | 9 | 9 | 28.8387
105_artifact | baseline | 0.7000 | 9 | 8 | 27.6427
105_artifact | baseline | 0.8500 | 9 | 0 | —
105_artifact | matched_filter | 0.5500 | 9 | 9 | 0.2516
105_artifact | matched_filter | 0.7000 | 9 | 9 | 0.2516
105_artifact | matched_filter | 0.8500 | 9 | 0 | —


## E10.5 — Odłożony test na trzech fragmentach rekordu 200

In [6]:
# CH10_LOCKED_TEST: pierwszy odczyt sygnału i adnotacji rekordu 200
freeze_path = root / "config/ch10_freeze.json"
commitment = json.loads(freeze_path.read_text(encoding="utf-8"))
for path, expected in commitment["files_sha256"].items():
    if checksum(root / path) != expected:
        raise ValueError(f"Plik protokołu zmienił się po zamrożeniu: {path}")
nb_path = root / "notebooks/ch10_validation_reproducibility.ipynb"
nb_code = ["".join(c["source"]) for c in json.loads(nb_path.read_text(encoding="utf-8"))["cells"]
           if c["cell_type"] == "code"]
if sha256(json.dumps(nb_code, ensure_ascii=False).encode()).hexdigest() != commitment["notebook_code_sha256"]:
    raise ValueError("Kod notebooka zmienił się po zamrożeniu.")
locked = load_locked(root, protocol)
test = {method: [] for method in methods}
by_fragment = {}
for fragment in locked:
    name = fragment["fragment_id"]
    pair = [evaluate(fragment, method, frozen, state) for method in methods]
    np.testing.assert_array_equal(pair[0][0].evaluation_mask, pair[1][0].evaluation_mask)
    for method, (result, item) in zip(methods, pair):
        test[method].append(item)
        by_fragment[name, method] = fragment, result
rows = [item["summary"] for items in test.values() for item in items]
summary = [aggregate(test[method]) for method in methods]
test_pairs = [p for items in test.values() for item in items for p in item["pairs"]]
test_windows = [w for items in test.values() for item in items for w in item["windows"]]
assert all(r["windows"] == 28 and r["reference_all"] == 396 for r in summary)
for name in protocol["locked_fragments"]:
    assert {r["split"] for r in rows if r["fragment_id"] == name} == {"test"}
checks = [check for r in summary for check in
          didactic_checks(r, protocol["didactic_targets_adapted_from_m8_E11"])]
for slug, table in (
    ("test_by_fragment", rows), ("test_summary", summary), ("test_pairs", test_pairs),
    ("test_windows", test_windows), ("didactic_checks", checks),
):
    write_csv(outputs / f"ch10_{slug}.csv", table)
show_rows(rows, ["fragment_id", "method", "tp", "fp", "fn", "f1",
                 "timing_mae_ms", "available_windows", "windows", "window_hr_mae_bpm"])
show_rows(summary, ["method", "sensitivity", "ppv", "f1", "timing_mae_ms", "timing_bias_ms",
                    "timing_p95_abs_ms", "rr_pairs", "rr_mae_ms", "window_hr_mae_bpm",
                    "result_coverage"])
show_rows([r for r in checks if not r["met"]], ["method", "metric", "value", "target"])

symbols = sorted({p["reference_symbol"] for p in test_pairs if p["reference_symbol"]})
strata = []
for method in methods:
    for symbol in symbols:
        refs = [p for p in test_pairs if p["method"] == method and p["reference_symbol"] == symbol]
        tp = sum(p["status"] == "TP" for p in refs)
        strata.append({"method": method, "reference_symbol": symbol, "references": len(refs),
                       "matched": tp, "sensitivity": tp / len(refs)})
write_csv(outputs / "ch10_test_symbols.csv", strata)
show_rows(strata, ["method", "reference_symbol", "references", "matched", "sensitivity"])

fig, axes = plt.subplots(1, 2, figsize=(8.4, 3.5), layout="constrained")
x = np.arange(len(locked))
labels = [f["fragment_id"].split("_", 1)[1] for f in locked]
for method, shift, marker in zip(methods, (-.08, .08), ("o", "s")):
    part = [r for r in rows if r["method"] == method]
    axes[0].scatter(x + shift, [r["sensitivity"] for r in part], marker=marker,
                    color=colors[method], label=LABELS[method])
    axes[1].scatter(x + shift, [r["result_coverage"] for r in part], marker=marker,
                    color=colors[method], label=LABELS[method])
for ax, label in zip(axes, ("Czułość QRS [–]", "Pokrycie HR [–]")):
    ax.set(xticks=x, xticklabels=labels, ylim=(-.04, 1.07),
           xlabel="Wycinek rekordu 200", ylabel=label)
axes[1].legend(loc="lower left")
save(fig, "ch10_04_test")

fragment_id | method | tp | fp | fn | f1 | timing_mae_ms | available_windows | windows | window_hr_mae_bpm
200_vt | baseline | 143 | 0 | 0 | 1.0000 | 0.7187 | 10 | 10 | 0.1481
200_noise | baseline | 126 | 0 | 0 | 1.0000 | 0.9480 | 9 | 9 | 0.1644
200_complex | baseline | 118 | 0 | 1 | 0.9958 | 0.4708 | 9 | 9 | 0.5930
200_vt | matched_filter | 143 | 0 | 0 | 1.0000 | 2.6224 | 10 | 10 | 0.1481
200_noise | matched_filter | 126 | 0 | 0 | 1.0000 | 2.5573 | 9 | 9 | 0.1644
200_complex | matched_filter | 119 | 0 | 0 | 1.0000 | 2.2176 | 9 | 9 | 0.1416
method | sensitivity | ppv | f1 | timing_mae_ms | timing_bias_ms | timing_p95_abs_ms | rr_pairs | rr_mae_ms | window_hr_mae_bpm | result_coverage
baseline | 0.9974 | 1.0000 | 0.9987 | 0.7178 | 0.1723 | 2.7778 | 383 | 1.4433 | 0.2963 | 1.0000
matched_filter | 1.0000 | 1.0000 | 1.0000 | 2.4771 | -1.7039 | 2.7778 | 385 | 4.9134 | 0.1513 | 1.0000
method | metric | value | target
method | reference_symbol | references | matched | sensitivity
baseline | A

## E10.6 — Rejestr niepowodzeń i kontrola wyników pośrednich

In [7]:
failures = []
fig, axes = plt.subplots(3, 2, figsize=(9.7, 6.4), layout="constrained", sharex="col")
for col, method in enumerate(methods):
    p = choose_failure(test[method], protocol["failure_selection"][method])
    fragment, result = by_fragment[p["fragment_id"], method]
    i = p["reference_sample"] if p["reference_sample"] is not None else p["detected_sample"]
    fs = float(fragment["metadata"]["fs_hz"])
    t = fragment["time_s"]
    local = np.abs(t - t[i]) <= 0.65
    shift = result.metadata.get("anchor_shift_samples", 0)
    score_t = t + shift / fs
    on_score = np.abs(score_t - t[i]) <= 0.70
    lo, hi = max(0, i - shift - 18), min(len(t), i - shift + 19)
    around = result.candidate_r_samples[np.abs(result.candidate_r_samples - i) <= 18]
    axes[0, col].plot(t[local], fragment["signal_mV"][local], lw=.95, color="#23374b")
    axes[1, col].plot(t[local], result.filtered[local], lw=.95, color="#23374b")
    axes[2, col].plot(score_t[on_score], result.score[on_score], lw=1, color=colors[method])
    axes[2, col].axhline(result.metadata["threshold_k_mad"], lw=.8, ls="--", color="#666666")
    for r in fragment["r_samples"][local[fragment["r_samples"]]]:
        axes[1, col].axvline(t[r], lw=.75, color="#875b34", alpha=.8)
    rr = result.r_samples[local[result.r_samples]]
    axes[1, col].scatter(t[rr], result.filtered[rr], s=23, facecolors="none", edgecolors=colors[method])
    for ax in axes[:, col]:
        ax.axvline(t[i], lw=1, ls=":", color="#212121")
        ax.set_xlim(t[i] - .60, t[i] + .60)
    axes[0, col].set_title(f"{LABELS[method].capitalize()} · {p['fragment_id']} · {p['status']}")
    axes[2, col].set_xlabel("Czas lokalny [s]")
    window = next((w for w in test_windows if w["fragment_id"] == p["fragment_id"]
                   and w["method"] == method and w["start_s"] <= t[i] < w["end_s"]), None)
    failures.append({
        **p, "local_time_s": float(t[i]), "record_time_s": float(t[i] + fragment["metadata"]["start_sample_record"] / fs),
        "largest_score_within_50ms_of_anchor": float(np.max(result.score[lo:hi])),
        "threshold": result.metadata["threshold_k_mad"],
        "n_candidates_within_50ms": len(around),
        "in_evaluation_mask": bool(result.evaluation_mask[i]),
        "window_start_s": window["start_s"] if window else None,
        "window_sqi": window["sqi"] if window else None,
        "window_result_available": window["result_available"] if window else None,
        "window_reason": window["missing_reason"] if window else "outside_complete_window",
    })
for ax, label in zip(axes[:, 0], ("EKG [mV]", "Po filtracji [mV]", "Wynik [–]")):
    ax.set_ylabel(label)
write_csv(outputs / "ch10_failure_log.csv", failures)
save(fig, "ch10_05_bledy")
show_rows(failures, ["method", "fragment_id", "status", "reference_symbol", "local_time_s",
                     "largest_score_within_50ms_of_anchor", "threshold", "n_candidates_within_50ms",
                     "window_result_available", "window_reason"])

method | fragment_id | status | reference_symbol | local_time_s | largest_score_within_50ms_of_anchor | threshold | n_candidates_within_50ms | window_result_available | window_reason
baseline | 200_complex | FN | V | 6.9500 | 1.3445 | 9.0000 | 0 | True | 
matched_filter | 200_vt | TP | V | 83.5278 | 10.6984 | 3.0000 | 1 | True | 


## E10.7 — Schemat tabeli okien i braki wartości

In [8]:
developed = [evaluate(f, m, frozen, state)[1] for m in methods for f in (*train, *validation)]
all_windows = [w for item in developed for w in item["windows"]] + test_windows
keys = [(w["record_id"], w["fragment_id"], w["method"], w["start_s"]) for w in all_windows]
assert len(keys) == len(set(keys))
assert all((w["hr_bpm"] is None) == (not w["result_available"]) for w in all_windows)
assert all(w["split"] == {"100": "train", "103": "train", "105": "validation", "200": "test"}[w["record_id"]]
           for w in all_windows)
write_csv(outputs / "ch10_windows_raw.csv", all_windows)
schema = {
    "unit_of_observation": "10 s window in a fragment for a fixed method",
    "key": ["record_id", "fragment_id", "method", "start_s"],
    "group": "record_id",
    "role": "split",
    "fields": {
        "hr_bpm": {"unit": "1/min", "missing": "empty if result unavailable"},
        "sqi": {"unit": "1", "role": "quality diagnostic, depends on detector"},
        "result_available": {"unit": "1", "role": "availability flag"},
        "missing_reason": {"role": "diagnostic string"},
        "valid_sample_fraction": {"unit": "1", "role": "input completeness"},
        "evaluation_sample_fraction": {"unit": "1", "role": "evaluation domain"},
        "reference_hr_bpm": {"unit": "1/min", "role": "comparison only"},
        "abs_hr_error_bpm": {"unit": "1/min", "role": "comparison only"},
        "start_s": {"unit": "s", "role": "fragment-local window start, inclusive"},
        "end_s": {"unit": "s", "role": "fragment-local window end, exclusive"},
    },
}
(outputs / "ch10_window_schema.json").write_text(
    json.dumps(schema, ensure_ascii=False, indent=2) + "\n", encoding="utf-8",
)
print(f"Zachowano {len(all_windows)} wierszy; brak częstości w {sum(not w['result_available'] for w in all_windows)}.")

Zachowano 162 wierszy; brak częstości w 0.


## E10.8 — Ślad wykonania

In [9]:
inputs = [
    protocol_path, freeze_path, root / "config/ch09_frozen.json", root / "data/ch09_run.json",
    root / "data/ch09_fitted_state.npz", root / "data/ch09_development_manifest.csv",
    root / "data/ch10_test_manifest.csv",
    *sorted((root / "data/test_locked").glob("*.*")),
    *sorted((root / "src/bspsbook").glob("*.py")),
]
result_files = sorted(p for p in outputs.glob("ch10_*") if p.name != "ch10_run.json")
run = {
    "environment": environment(),
    "protocol": protocol["protocol_version"],
    "notebook_code_sha256": commitment["notebook_code_sha256"],
    "inputs_sha256": {str(p.relative_to(root)): checksum(p) for p in inputs},
    "seeds": seeds,
    "fit_records": state["fit_records"],
    "selection_records": ["105"],
    "test_record": "200",
    "test_fragments": protocol["locked_fragments"],
    "test_status": "executed_frozen_configuration",
    "test_summary": summary,
    "results_sha256": {str(p.relative_to(root)): checksum(p) for p in
                       (*result_files, *sorted(figures.glob("ch10_*")))},
}
(outputs / "ch10_run.json").write_text(
    json.dumps(run, indent=2, ensure_ascii=False, allow_nan=False) + "\n", encoding="utf-8",
)
print(f"Zweryfikowano ślad wykonania {protocol['protocol_version']}; protokół SHA-256 {checksum(protocol_path)}.")

Zweryfikowano ślad wykonania ch10-v0.2-D01; protokół SHA-256 3fbbdd91a850f32ef2dcdec24bd4932e564d3d6fb117f16bf4b6e505a1e4a96a.
